# 訓練、驗證與最佳權重

對應原作業：Day 71、73–80、85–87。這是現代 PyTorch 練習；原始作業保留在 `../../homework/`。

從上到下執行即可。預設使用 CPU、合成資料，不需下載外部資料；改為真實資料時，先決定資料粒度與切分方式。

## 核心順序
`zero_grad` → forward → loss → `backward` → `step`。驗證時 `eval()` + `inference_mode()`，不更新參數。早停由驗證集決定，測試集只在最後使用。完整可重用腳本見 `../../examples/torch_training.py`。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X = torch.randn(600, 2)
y = (X[:, 0] + 0.6 * X[:, 1] > 0).long()
train, valid = random_split(TensorDataset(X, y), [480, 120], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train, batch_size=48, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid, batch_size=120, num_workers=0)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)


In [ ]:
history = []
for epoch in range(1, 11):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    correct, total, val_loss = 0, 0, 0.0
    with torch.inference_mode():
        for xb, yb in valid_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            val_loss += criterion(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    history.append((epoch, val_loss / total, correct / total))

print("last epoch / val loss / val accuracy:", history[-1])
assert history[-1][2] > 0.85
